In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%pip install anomalib

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 246.1/246.1 kB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 59.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 848.6/848.6 kB 63.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 53.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 28.1/28.1 MB 76.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 136.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 800.2/800.2 kB 45.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 852.4/852.4 kB 62.3 MB/s eta 0:00:00
  Created wheel for freia: filename=FrEIA-0.2-py3-none-any.whl size=42763 sha256=a674a681227f4505c4f4a3d335ed078d5607a4dc4c231360aff0ebd6ec7e1c29
  Stored in direc

In [ ]:
import torch
from torch import nn
from anomalib.data import MVTecAD  #anomalib MVtecad dataset
from anomalib.engine import Engine
from torchvision import transforms
engine = Engine(
    accelerator="gpu",
    devices=1
)
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor()
])

ModuleNotFoundError: No module named 'anomalib'

In [ ]:
#USING THE INTEL ANOMALIB MVTECAD DATASET AND DATALOADER

data = MVTecAD(root="/content/drive/MyDrive",train_batch_size=8,
    eval_batch_size=8 )
data.setup()
print(data.__dict__.keys())

dict_keys(['_log_hyperparams', 'prepare_data_per_node', 'allow_zero_length_dataloader_with_multiple_devices', 'trainer', 'train_batch_size', 'eval_batch_size', 'num_workers', 'test_split_mode', 'test_split_ratio', 'val_split_mode', 'val_split_ratio', 'seed', 'train_augmentations', 'val_augmentations', 'test_augmentations', '_samples', '_category', '_is_setup', 'external_collate_fn', 'root', 'train_data', 'test_data', 'val_data'])


In [ ]:
print(len(data.train_data))
print(len(data.test_data))
print(len(data.val_data))

209
83
83


In [ ]:
from anomalib.data.dataclasses.torch.base import Batch
loader=data.train_dataloader()

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


In [ ]:
# AUTOENCODER MODEL HAVING ENCODER AND DECODER SETUP WITH CNNS

class Autoencoder(nn.Module):
    def __init__(self):
        super().__init__()
        # Encoder Convolutional Layers
        self.encoder_conv = nn.Sequential(
            nn.Conv2d(3, 32, 4, stride=2, padding=1),   # 256x256 -> 128x128
            nn.ReLU(),
            nn.Conv2d(32, 64, 4, stride=2, padding=1),  # 128x128 -> 64x64
            nn.ReLU(),
            nn.Conv2d(64, 128, 4, stride=2, padding=1), # 64x64 -> 32x32
            nn.ReLU(),
            nn.Conv2d(128, 256, 4, stride=2, padding=1) # 32x32 -> 16x16
        )

        # Tight Linear Bottleneck to force semantic compression (improved the results)
        self.flatten = nn.Flatten()
        self.fc_encode = nn.Linear(256 * 16 * 16, 128)   # Latent vector dimension = 128

        # Decoder Linear Layers
        self.fc_decode = nn.Linear(128, 256 * 16 * 16)
        self.unflatten = nn.Unflatten(1, (256, 16, 16))

        # Decoder Convolutional Layers
        self.decoder_conv = nn.Sequential(
            nn.ConvTranspose2d(256, 128, 4, stride=2, padding=1), # 16x16 -> 32x32
            nn.ReLU(),
            nn.ConvTranspose2d(128, 64, 4, stride=2, padding=1),  # 32x32 -> 64x64
            nn.ReLU(),
            nn.ConvTranspose2d(64, 32, 4, stride=2, padding=1),   # 64x64 -> 128x128
            nn.ReLU(),
            nn.ConvTranspose2d(32, 3, 4, stride=2, padding=1),    # 128x128 -> 256x256
            nn.Sigmoid()
        )

    def forward(self, x):
        # Pass through convolutional encoder
        x = self.encoder_conv(x)
        # Flatten and compress into tight bottleneck
        x = self.flatten(x)
        bottleneck = self.fc_encode(x)

        # Decompress out of bottleneck
        x = self.fc_decode(bottleneck)
        x = self.unflatten(x)
        # Pass through convolutional decoder
        decode = self.decoder_conv(x)
        return decode

In [ ]:
from torch.optim import Adam

In [ ]:
encoder=Autoencoder()
loss=nn.MSELoss()

optimizer=Adam(encoder.parameters(),lr=0.001)

In [ ]:
import torch.nn.functional as F

encoder = Autoencoder().to(device) # Move model to GPU
loss_fn = nn.MSELoss()
optimizer = Adam(encoder.parameters(), lr=0.001)

epochs = 100
for epoch in range(epochs):
    total_loss = 0
    encoder.train()
    for batch_idx, batch in enumerate(loader):
        optimizer.zero_grad()

        # Correctly interpolate on device
        inputs = batch.image.to(device)
        inputs = F.interpolate(inputs, size=(256, 256))

        output = encoder(inputs)
        loss_ = loss_fn(output, inputs)
        loss_.backward()
        optimizer.step()

        total_loss += loss_.item()

    avg_loss = total_loss / len(loader)
    print(f"Epoch: {epoch+1}/{epochs} | Avg Loss: {avg_loss:.6f}")

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


Epoch: 1/100 | Avg Loss: 0.034577
Epoch: 2/100 | Avg Loss: 0.004067
Epoch: 3/100 | Avg Loss: 0.003074
Epoch: 4/100 | Avg Loss: 0.002848
Epoch: 5/100 | Avg Loss: 0.002681
Epoch: 6/100 | Avg Loss: 0.002611
Epoch: 7/100 | Avg Loss: 0.002592
Epoch: 8/100 | Avg Loss: 0.002561
Epoch: 9/100 | Avg Loss: 0.002477
Epoch: 10/100 | Avg Loss: 0.002425
Epoch: 11/100 | Avg Loss: 0.002479
Epoch: 12/100 | Avg Loss: 0.002455
Epoch: 13/100 | Avg Loss: 0.002376
Epoch: 14/100 | Avg Loss: 0.002407
Epoch: 15/100 | Avg Loss: 0.002348
Epoch: 16/100 | Avg Loss: 0.002391
Epoch: 17/100 | Avg Loss: 0.002486
Epoch: 18/100 | Avg Loss: 0.002338
Epoch: 19/100 | Avg Loss: 0.002371
Epoch: 20/100 | Avg Loss: 0.002425
Epoch: 21/100 | Avg Loss: 0.002343
Epoch: 22/100 | Avg Loss: 0.002281
Epoch: 23/100 | Avg Loss: 0.002331
Epoch: 24/100 | Avg Loss: 0.002300
Epoch: 25/100 | Avg Loss: 0.002420
Epoch: 26/100 | Avg Loss: 0.002321
Epoch: 27/100 | Avg Loss: 0.002288
Epoch: 28/100 | Avg Loss: 0.002305
Epoch: 29/100 | Avg Loss: 0.0

In [ ]:
encoder.eval()

train_loader = data.train_dataloader()

with torch.no_grad():
    batch = next(iter(train_loader))

    imgs = batch.image[:8].to(device)
    imgs = F.interpolate(imgs, size=(256,256))

    recon = encoder(imgs)

fig, axes = plt.subplots(2, 8, figsize=(15,4))

for i in range(8):

    orig = imgs[i].cpu().permute(1,2,0)
    rec = recon[i].cpu().permute(1,2,0)

    axes[0,i].imshow(orig)
    axes[0,i].axis("off")

    axes[1,i].imshow(rec)
    axes[1,i].axis("off")

axes[0,0].set_title("Train Original")
axes[1,0].set_title("Train Reconstruction")

plt.tight_layout()
plt.show()

NameError: name 'plt' is not defined

In [ ]:
test_loader=data.test_dataloader()

In [ ]:
for batch in test_loader:
    print(batch)
    print(batch.__dict__.keys())
    break

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Set model to evaluation mode
encoder.eval()

test_loss = 0
normal_scores = []
defect_scores = []

# We will collect a few sample images during evaluation to visualize later
sample_originals = []
sample_reconstructions = []
sample_labels = []

with torch.no_grad():
    for batch in test_loader:
        # 1. Correctly move test data to device and resize
        inputs = batch.image.to(device)
        inputs = F.interpolate(inputs, size=(256, 256))
        labels = batch.gt_label

        # 2. Forward pass to get reconstructions
        outputs = encoder(inputs)

        # 3. Calculate global test loss for tracking
        batch_loss = loss_fn(outputs, inputs)
        test_loss += batch_loss.item()

        # 4. Calculating per-pixel MSE
        # Shape: [Batch, 256, 256]
        error_maps = ((inputs - outputs) ** 2).mean(dim=1)

        # Store scores based on true anomaly label (0 = Normal, 1 = Defective)
        for i in range(len(error_maps)):
            max_local_error = error_maps[i].max().item()
            if labels[i] == 0:
                normal_scores.append(max_local_error)
            else:
                defect_scores.append(max_local_error)

        # Save the first batch's worth of images for plotting
        if len(sample_originals) < 4:
            sample_originals.extend(inputs.cpu())
            sample_reconstructions.extend(outputs.cpu())
            sample_labels.extend(labels.numpy())

# Calculate and print overall test metrics
avg_test_loss = test_loss / len(test_loader)
print("==============================")
print(f"Test Dataset Avg Batch Loss: {avg_test_loss:.6f}")
print(f"Train Dataset Avg Batch Loss: {avg_loss:.6f}")

print("==============================")
if normal_scores:
    print(f"Train Samples Max Local Error Range: {min(normal_scores):.4f} -> {max(normal_scores):.4f}")
if defect_scores:
    print(f"Defective Samples Max Local Error Range: {min(defect_scores):.4f} -> {max(defect_scores):.4f}")
print("==============================\n")


# 5. VISUALIZATION CODE: Plot original vs reconstruction vs anomaly map
num_samples = min(4, len(sample_originals))
fig, axes = plt.subplots(num_samples, 3, figsize=(12, 3 * num_samples))

# Fix dimensions if only 1 sample is displayed
if num_samples == 1:
    axes = np.expand_dims(axes, axis=0)

for idx in range(num_samples):
    orig = sample_originals[idx].permute(1, 2, 0).numpy()
    recon = sample_reconstructions[idx].permute(1, 2, 0).numpy()

    # Generate the structural discrepancy heatmap
    diff_map = np.mean((orig - recon) ** 2, axis=-1)

    label_text = "Anomaly" if sample_labels[idx] != 0 else "Normal"

    # Column 1: Original Image
    axes[idx, 0].imshow(orig)
    axes[idx, 0].set_title(f"Original ({label_text})")
    axes[idx, 0].axis("off")

    # Column 2: Network Reconstruction
    axes[idx, 1].imshow(recon)
    axes[idx, 1].set_title("Reconstruction")
    axes[idx, 1].axis("off")

    # Column 3: Error Heatmap (Bright areas show where the defect is caught)
    heatmap = axes[idx, 2].imshow(diff_map, cmap="jet")
    axes[idx, 2].set_title("Defect Map (Discrepancy)")
    axes[idx, 2].axis("off")
    fig.colorbar(heatmap, ax=axes[idx, 2], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))

plt.hist(normal_scores,
         bins=20,
         alpha=0.6,
         label="Normal")

plt.hist(defect_scores,
         bins=20,
         alpha=0.6,
         label="Defective")

plt.xlabel("Reconstruction Error")
plt.ylabel("Number of Images")
plt.title("Normal vs Defective Error Distribution")
plt.legend()

plt.show()

In [ ]:
plt.figure(figsize=(6,5))

plt.boxplot(
    [normal_scores, defect_scores],
    labels=["Normal", "Defective"]
)

plt.ylabel("Maximum Reconstruction Error")
plt.title("Error Distribution Comparison")

plt.show()

In [ ]:
!pip install -q gradio opencv-python

import gradio as gr
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
import cv2

# Set evaluation mode
encoder.eval()

def inspect_anomaly(input_image, threshold):
    if input_image is None:
        return None, None, None, "Please upload an image."

    # 1. Preprocess input
    img_resized = cv2.resize(input_image, (256, 256))
    img_norm = img_resized.astype(np.float32) / 255.0
    tensor_input = torch.tensor(img_norm).permute(2, 0, 1).unsqueeze(0).to(device)

    # 2. Model Inference
    with torch.no_grad():
        reconstruction = encoder(tensor_input)

    # 3. Postprocess outputs
    recon_np = reconstruction.squeeze(0).cpu().permute(1, 2, 0).numpy()
    orig_np = tensor_input.squeeze(0).cpu().permute(1, 2, 0).numpy()

    # Per-pixel MSE across color channels
    error_map = np.mean((orig_np - recon_np) ** 2, axis=-1)
    max_error = float(np.max(error_map))
    mean_error = float(np.mean(error_map))

    # Normalize error map for heatmap visualization
    norm_map = (error_map - error_map.min()) / (error_map.max() - error_map.min() + 1e-8)
    heatmap_colored = cv2.applyColorMap(np.uint8(255 * norm_map), cv2.COLORMAP_JET)
    heatmap_colored = cv2.cvtColor(heatmap_colored, cv2.COLOR_BGR2RGB)

    # Blend heatmap over original image
    overlay = cv2.addWeighted((orig_np * 255).astype(np.uint8), 0.6, heatmap_colored, 0.4, 0)

    # Anomaly mask based on threshold slider
    anomaly_mask = (error_map > threshold).astype(np.uint8) * 255

    status = "DEFECT DETECTED" if max_error > threshold else "NORMAL"
    metrics = f"Status: {status}\nMax Pixel Error: {max_error:.5f}\nMean Error: {mean_error:.5f}\nCurrent Threshold: {threshold:.5f}"

    return (recon_np * 255).astype(np.uint8), overlay, anomaly_mask, metrics

# Build Gradio UI
with gr.Blocks(title="MVTec Anomaly Detection Demo") as demo:
    gr.Markdown("# Visual Anomaly Inspection with Autoencoders")
    gr.Markdown("Upload an industrial part inspection image to detect structural defects and anomalies.")

    with gr.Row():
        with gr.Column():
            input_img = gr.Image(type="numpy", label="Input Inspection Image")
            threshold_slider = gr.Slider(
                minimum=0.01,
                maximum=0.50,
                value=0.08,
                step=0.005,
                label="Anomaly Decision Threshold"
            )
            run_btn = gr.Button("Inspect Image", variant="primary")
            results_box = gr.Textbox(label="Detection Results", interactive=False)

        with gr.Column():
            out_recon = gr.Image(label="Model Reconstruction", type="numpy")
            out_overlay = gr.Image(label="Defect Heatmap Overlay", type="numpy")
            out_mask = gr.Image(label="Binary Defect Mask", type="numpy")

    run_btn.click(
        fn=inspect_anomaly,
        inputs=[input_img, threshold_slider],
        outputs=[out_recon, out_overlay, out_mask, results_box]
    )

# share=True creates a public 72-hour link to share with anyone
demo.launch(share=True, debug=True)